# **Noise2Noise (2D) — powered by CAREamics**

---

<font size = 4> Noise2Noise is a deep-learning method that denoises images by training on **pairs of independently-noisy images of the same scene** — no clean, high-quality ground truth is required. It was originally published by [Lehtinen *et al.* (2018)](https://arxiv.org/abs/1803.04189). The network learns to map one noisy image to a second, independent noisy acquisition of the same content; because the two noise realisations are independent, the network converges to the underlying clean signal.

<font size = 4> **This notebook runs Noise2Noise on 2D datasets using [CAREamics](https://careamics.github.io/), a modern PyTorch/Lightning implementation.**

---

<font size = 4>*Disclaimer*:

<font size = 4>This notebook is part of the Zero-Cost Deep-Learning to Enhance Microscopy project (https://github.com/HenriquesLab/DeepLearning_Collab/wiki). Jointly developed by the Jacquemet (https://cellmig.org/) and Henriques (https://henriqueslab.github.io/) laboratories.

<font size = 4>The deep-learning engine used here is **CAREamics** (https://github.com/CAREamics/careamics).

<font size = 4>This notebook is based on:

<font size = 4>**Noise2Noise: Learning Image Restoration without Clean Data**, Lehtinen *et al.*, ICML 2018 (https://arxiv.org/abs/1803.04189)

<font size = 4>**Please cite the original Noise2Noise paper and CAREamics when using this notebook.**

# **How to use this notebook?**

---

<font size = 4>This notebook is structured in numbered sections. Run the cells from top to bottom.

---
### **Structure of a notebook**

<font size = 4>**Text cells** provide information. **Code cells** contain code; move your cursor over the `[ ]` on the left and click the play button to execute.

---
### **Making changes to the notebook**

<font size = 4>**Make a copy** of this notebook and save it to your Google Drive (`File -> Save a copy in Drive`) before editing.

# **0. Before getting started**
---

<font size = 4>Before running the notebook, make sure you are logged into your Google account and that your data is in your Google Drive.

<font size = 4>Noise2Noise requires **paired training data**: for each field of view you need **two independently-acquired noisy images** of the same scene. One acts as the input, the other as the target.

<font size = 4>Please note that you can **only use .tif or .tiff files!**

<font size = 4>The input and target images must be provided in **two separate folders**, and paired images must have **matching file names**.

<font size = 4>A common data structure that works well:

*   Data
    - **Training**
        - source (noisy) — img_1.tif, img_2.tif ...
        - target (independently noisy, same scenes) — img_1.tif, img_2.tif ...
    - **Quality control** (optional but recommended)
        - Low SNR images — img_1.tif, img_2.tif ...
        - High SNR images — img_1.tif, img_2.tif ...
    - **Prediction** — images to denoise
    - **Results**

---
<font size = 4>**Important note**

<font size = 4>- To **train from scratch**: run **sections 1-4**, then **section 5** to assess quality and **section 6** to predict.
<font size = 4>- To **continue training from a checkpoint**: run **sections 1-4** with `Use_pretrained_model` enabled and training paths filled in.
<font size = 4>- To only **run predictions with an existing checkpoint**: run **sections 1-3**, enable `Use_pretrained_model`, run **section 4.1** to load the model, skip **section 4.2**, then run **section 6**.
---


# **1. Install CAREamics and dependencies**
---

## **1.1. Install CAREamics**

In [ ]:
#@markdown ##Install CAREamics and dependencies
#@markdown This installs a pinned, tested version of CAREamics. It takes several minutes,
#@markdown because CAREamics 0.3.2 requires torch < 2.10 and Colab ships a newer one, so
#@markdown PyTorch and its CUDA libraries are reinstalled.

# NumPy is deliberately NOT pinned. CAREamics 0.3.2 accepts numpy<=2.4.6, which the
# version Colab already ships satisfies, so pip leaves it alone. Pinning it lower would
# force pip to build NumPy from source on Colab's Python 3.13, which is slow and fragile.
!pip install "careamics==0.3.2" "careamics-portfolio" -q

print("CAREamics installed.")

## **1.2. Restart the runtime (only if you see an import error)**
<font size = 4>The install above keeps Colab's existing NumPy, so you can normally continue straight to section 1.3. **If section 1.3 raises a NumPy or import error**, go to `Runtime -> Restart session`, then re-run from section 1.3 (do **not** re-run the install cell).

## **1.3. Load key dependencies**

In [ ]:
#@markdown ##Load key dependencies
import csv
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import tifffile

import careamics
from careamics import CAREamist
from careamics.config import create_n2n_config

# Both metrics below are scale invariant: the denoised prediction, the noisy input and
# the high-SNR ground truth are not guaranteed to share an intensity scale, and a plain
# PSNR or SSIM would largely measure that difference rather than image quality.
# `scale_invariant_psnr` is not re-exported by `careamics.metrics`, so it has to be
# imported from the submodule. `microssim` is the SSIM implementation CAREamics itself
# depends on, and it is installed alongside CAREamics.
from careamics.metrics.metrics import scale_invariant_psnr
from microssim import micro_structural_similarity

TIFF_SUFFIXES = {".tif", ".tiff"}


def require_path(path_value, field_name, must_exist=True):
    """Return `path_value` as a Path, with a clear error if it is unset or missing.

    An empty string would otherwise become `Path(".")`, which silently points at the
    whole Colab working directory.
    """
    path_text = str(path_value).strip()
    if not path_text:
        raise ValueError(f"Please set `{field_name}` before running this cell.")

    path = Path(path_text).expanduser()
    if must_exist and not path.exists():
        raise FileNotFoundError(f"`{field_name}` does not exist: {path}")
    return path


print(f"CAREamics version: {careamics.__version__}")


# **2. Initialise the Colab session**
---

## **2.1. Check for GPU access**

In [ ]:
#@markdown ##Run this cell to check if you have GPU access
import torch

if torch.cuda.is_available():
    print("You have GPU access.")
    print(torch.cuda.get_device_name(0))
else:
    print("You do NOT have GPU access.")
    print("Go to 'Runtime -> Change runtime type' and select a GPU hardware accelerator,")
    print("then re-run the notebook. Expect slow performance on CPU.")

## **2.2. Mount your Google Drive**

In [ ]:
#@markdown ##Play the cell to connect your Google Drive to Colab
#@markdown * Follow the instructions.
#@markdown * Click on "Files" on the left. Refresh it — your Google Drive appears as "drive".

from google.colab import drive
drive.mount('/content/gdrive')

## **2.3. (Optional) Download an example dataset**
<font size = 4>If you just want to try the notebook without your own data, run this cell to download the **SEM** Noise2Noise example dataset, the same one used in the [CAREamics Noise2Noise example](https://github.com/CAREamics/careamics-examples/blob/main/applications/n2n/2D/SEM_Noise2Noise.ipynb).

<font size = 4>The dataset contains a `train` and a `test` image, each a stack of the same scene acquired at 7 increasing scan times. Two independent 1 us acquisitions form the Noise2Noise pair, and the 5 us averaged acquisition serves as a **pseudo ground truth** for quality control.

<font size = 4>Training uses the `train` image; quality control and prediction use the **held-out `test` image**, so the metrics in section 5.2 tell you how well the model generalises rather than how well it fits the data it was trained on.

<font size = 4>The printed paths can be pasted into the fields in section 3.

In [ ]:
#@markdown ##(Optional) Download the SEM example dataset for testing
Download_example_dataset = True #@param {type:"boolean"}

if Download_example_dataset:
    from careamics_portfolio import PortfolioManager

    portfolio = PortfolioManager()
    download = portfolio.denoising.N2N_SEM.download("./example_data_n2n_download")
    example_tiffs = {
        Path(f).stem: Path(f) for f in download
        if Path(f).suffix.lower() in TIFF_SUFFIXES
    }
    missing = {"train", "test"} - example_tiffs.keys()
    if missing:
        raise RuntimeError(
            f"The N2N SEM download is missing {sorted(missing)}. "
            f"Found: {sorted(example_tiffs)}."
        )

    stacks = {}
    for name in ("train", "test"):
        stack = tifffile.imread(example_tiffs[name])
        if stack.ndim != 3 or stack.shape[0] < 4:
            raise RuntimeError(
                f"Unexpected shape for the SEM {name} stack: {stack.shape}."
            )
        stacks[name] = stack

    example_root = Path("example_data_n2n")
    example_folders = {
        "Training_source": example_root / "input",
        "Training_target": example_root / "target",
        "Source_QC_folder": example_root / "qc_source",
        "Target_QC_folder": example_root / "qc_target",
        "Data_folder": example_root / "prediction",
        "Result_folder": example_root / "results",
        "model_path": example_root / "model",
    }
    for folder in example_folders.values():
        folder.mkdir(parents=True, exist_ok=True)

    # Frames 2 and 3 are two independent 1 us scans of the same scene, so they form a
    # Noise2Noise pair. The last frame (5 us, averaged over 4 acquisitions) is clean
    # enough to act as a pseudo ground truth.
    tifffile.imwrite(
        example_folders["Training_source"] / "scene_00.tif", stacks["train"][2])
    tifffile.imwrite(
        example_folders["Training_target"] / "scene_00.tif", stacks["train"][3])

    # Quality control and prediction use the held-out test image.
    tifffile.imwrite(
        example_folders["Source_QC_folder"] / "scene_00.tif", stacks["test"][2])
    tifffile.imwrite(
        example_folders["Target_QC_folder"] / "scene_00.tif", stacks["test"][-1])
    tifffile.imwrite(
        example_folders["Data_folder"] / "scene_00.tif", stacks["test"][2])

    print("Training pair from 'train', quality control and prediction from 'test'.")
    print("Paste these paths into the fields below:\n")
    for field, folder in example_folders.items():
        print(f"  {field:18} {folder}")


# **3. Select your parameters and paths**
---

## **3.1. Setting the main training parameters**
<font size = 4>These fields are **pre-filled with the paths printed by section 2.3**, so you can run the example dataset straight through. **Replace them with your own paths** when you use your own data.

<font size = 4>`Training_source` and `Training_target` should each point to a folder of `.tif` or `.tiff` images, or to a single TIFF file. Paired source and target files must have **matching file names**.

<font size = 4>`model_path` is where the trained model, its checkpoints and the quality control results are written. For your own data, point it at a folder **on your Google Drive**, otherwise everything is lost when the Colab runtime shuts down.

<font size = 4>**Prediction tile size** is used in sections 5.2 and 6.1. Large images are denoised tile by tile to keep memory use low. Reduce it if you run out of GPU memory; it must be at least 128, because tiles overlap by 48 pixels.

<font size = 4>**Patch size** must be divisible by 8 and should be smaller than the smallest dimension of your images. **Batch size** is the number of patches seen per training step; lower it if you run out of GPU memory.

In [ ]:
#@markdown ###Path to the noisy input images (folder of .tif/.tiff files, or a single file):
Training_source = "example_data_n2n/input" #@param {type:"string"}
#@markdown ###Path to the independently-noisy target images (folder of .tif/.tiff files, or a single file):
Training_target = "example_data_n2n/target" #@param {type:"string"}

#@markdown ###Model name and output folder (use a folder on your Google Drive for your own data):
model_name = "my_n2n_model" #@param {type:"string"}
model_path = "example_data_n2n/model" #@param {type:"string"}

#@markdown ###Training parameters
#@markdown Number of epochs:
number_of_epochs = 100 #@param {type:"number"}
#@markdown Patch size (pixels, square, divisible by 8):
patch_size = 64 #@param {type:"number"}
#@markdown Batch size:
batch_size = 64 #@param {type:"number"}
#@markdown Number of patches held out for validation:
n_val_patches = 8 #@param {type:"number"}

#@markdown ###Prediction parameters
#@markdown Tile size used for prediction in sections 5.2 and 6.1 (pixels, square, at least 128):
prediction_tile_size = 256 #@param {type:"number"}


## **3.2. Data augmentation**
<font size = 4>Data augmentation (flips and 90° rotations) usually improves results and is recommended.

In [ ]:
#@markdown ##Enable or disable data augmentation:
Use_Data_augmentation = True #@param {type:"boolean"}

## **3.3. Using a pre-trained model**
<font size = 4>You can continue training from a previously trained CAREamics model. Provide the path to a checkpoint (`.ckpt`). The pre-trained model's configuration is reused, so the parameters above are ignored when this is enabled.

In [ ]:
#@markdown ##Load weights from a pre-trained CAREamics model
Use_pretrained_model = False #@param {type:"boolean"}
#@markdown ###If enabled, provide the path to the checkpoint (.ckpt) file:
pretrained_model_path = "" #@param {type:"string"}

# **4. Train the network**
---

## **4.1. Prepare the training data and model**

In [ ]:
#@markdown ##Create the configuration and the CAREamist
# Augmentations: None -> default (flips + 90-degree rotations); [] -> disabled
augmentations = None if Use_Data_augmentation else []

work_dir = require_path(model_path, "model_path", must_exist=False)
work_dir.mkdir(parents=True, exist_ok=True)

if Use_pretrained_model:
    checkpoint_path = require_path(pretrained_model_path, "pretrained_model_path")
    print(f"Loading pre-trained model from: {checkpoint_path}")
    careamist = CAREamist(checkpoint_path=checkpoint_path, work_dir=work_dir)
else:
    config = create_n2n_config(
        experiment_name=model_name,
        data_type="tiff",
        axes="YX",
        patch_size=(patch_size, patch_size),
        batch_size=batch_size,
        num_epochs=number_of_epochs,
        n_val_patches=n_val_patches,
        augmentations=augmentations,
    )
    print(config)
    careamist = CAREamist(config, work_dir=work_dir)


## **4.2. Start training**
<font size = 4>Training checkpoints are saved automatically to your output folder. If you loaded a pre-trained model and only want to run predictions, leave the training paths empty and skip this section.


In [ ]:
#@markdown ##Start training
training_source_text = str(Training_source).strip()
training_target_text = str(Training_target).strip()

if not training_source_text or not training_target_text:
    if Use_pretrained_model:
        print("No training paths provided. Keeping the loaded model for evaluation/prediction.")
    else:
        missing = []
        if not training_source_text:
            missing.append("Training_source")
        if not training_target_text:
            missing.append("Training_target")
        raise ValueError("Please set " + " and ".join(missing) + " before training.")
else:
    careamist.train(
        train_data=require_path(Training_source, "Training_source"),
        train_data_target=require_path(Training_target, "Training_target"),
    )
    print("Training complete.")


# **5. Evaluate your model**
---

## **5.1. Inspection of the loss function**

In [ ]:
#@markdown ##Plot the training and validation loss vs. epoch
try:
    loss_dict = careamist.get_losses()
except Exception as error:
    raise RuntimeError(
        "Could not read training losses. This section is only available after "
        "training in the current work_dir."
    ) from error

if not loss_dict.get("train_loss"):
    raise RuntimeError("No training loss values were found in the CSV logs.")

plt.figure(figsize=(8, 5))
plt.plot(loss_dict["train_epoch"], loss_dict["train_loss"], label="Train loss")
if loss_dict.get("val_loss"):
    plt.plot(loss_dict["val_epoch"], loss_dict["val_loss"], label="Validation loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.title("Training losses")
plt.show()


## **5.2. Error mapping and quality metrics estimation**
---
<font size = 4>This section evaluates the trained model on a **Quality Control dataset**: pairs of low-SNR images (`Source_QC_folder`) and matching high-SNR images (`Target_QC_folder`). Paired files must have **the same file name**.

<font size = 4>For the example dataset these are the **held-out `test` image**, which the model never saw during training, so the scores describe how well the model *generalises*.

<font size = 4>Each metric is computed twice: once for the **prediction against the ground truth**, and once for the **noisy input against the ground truth**. The second is a baseline. If denoising helped, the prediction scores better than the input, and the gap between the two is the improvement the model gave you.

<font size = 4>**SI-PSNR (scale-invariant peak signal-to-noise ratio)**, in decibels: how closely an image matches the ground truth. Higher is better, and roughly +3 means the error was halved.

<font size = 4>**MicroSSIM**: a structural similarity index adapted to microscopy. It asks whether the same *structures* are present rather than whether pixel values match, and runs from 0 to 1, where 1 is a perfect match.

<font size = 4>Both are **scale invariant**, so they are unaffected by the prediction, the noisy acquisition and the high-SNR acquisition sitting on different intensity scales. This is the same metric used in the [CAREamics Noise2Noise example](https://github.com/CAREamics/careamics-examples/blob/main/applications/n2n/2D/SEM_Noise2Noise.ipynb).

<font size = 4>The figure shows the noisy input, the prediction and the ground truth, with a zoomed-in crop of each underneath so you can judge fine detail. Per-image scores, and a MicroSSIM error map for each image, are written to a `Quality Control` folder next to your model, along with `QC_metrics_<model_name>.csv`.

In [ ]:
#@markdown ##Provide the Quality Control folders (paired low-SNR and high-SNR .tif/.tiff images)
Source_QC_folder = "example_data_n2n/qc_source" #@param {type:"string"}
Target_QC_folder = "example_data_n2n/qc_target" #@param {type:"string"}
#@markdown Size of the zoomed-in crop shown under the full images (pixels, square):
qc_crop_size = 256 #@param {type:"number"}

source_qc_path = require_path(Source_QC_folder, "Source_QC_folder")
target_qc_path = require_path(Target_QC_folder, "Target_QC_folder")

# ZeroCostDL4Mic convention: QC results live in a "Quality Control" folder next to the
# model they describe.
qc_dir = work_dir / model_name / "Quality Control"
qc_dir.mkdir(parents=True, exist_ok=True)

# Let CAREamics list the source files itself, so that each prediction stays paired with
# the file it came from.
qc_predictions, qc_sources = careamist.predict(
    pred_data=source_qc_path,
    tile_size=(prediction_tile_size, prediction_tile_size),
)

csv_path = qc_dir / f"QC_metrics_{model_name}.csv"
scores = {"si_psnr_pred": [], "si_psnr_input": [], "ssim_pred": [], "ssim_input": []}
first_image = None  # kept for the figure; the rest are not held in memory

with open(csv_path, "w", newline="") as csv_file:
    writer = csv.writer(csv_file)
    writer.writerow([
        "image",
        "Prediction v. GT SI-PSNR",
        "Input v. GT SI-PSNR",
        "Prediction v. GT MicroSSIM",
        "Input v. GT MicroSSIM",
    ])

    for prediction, source in zip(qc_predictions, qc_sources):
        source_file = Path(source)
        target_file = target_qc_path / source_file.name
        if not target_file.exists():
            raise FileNotFoundError(
                f"No ground truth named '{source_file.name}' in {target_qc_path}. "
                "Paired QC images must have matching file names."
            )

        gt = tifffile.imread(target_file).astype(np.float32)
        noisy = tifffile.imread(source_file).astype(np.float32)
        denoised = np.asarray(prediction).squeeze().astype(np.float32)

        si_psnr_pred = scale_invariant_psnr(gt, denoised)
        si_psnr_input = scale_invariant_psnr(gt, noisy)
        ssim_pred = micro_structural_similarity(gt, denoised)
        ssim_input = micro_structural_similarity(gt, noisy)

        writer.writerow([source_file.name, si_psnr_pred, si_psnr_input,
                         ssim_pred, ssim_input])
        scores["si_psnr_pred"].append(si_psnr_pred)
        scores["si_psnr_input"].append(si_psnr_input)
        scores["ssim_pred"].append(ssim_pred)
        scores["ssim_input"].append(ssim_input)

        # Per-pixel MicroSSIM error map, saved for inspection in Fiji or napari.
        error_map = micro_structural_similarity(
            gt, denoised, return_individual_components=True
        ).SSIM.astype(np.float32)
        tifffile.imwrite(qc_dir / f"MicroSSIM_GTvsPrediction_{source_file.name}", error_map)

        if first_image is None:
            first_image = (noisy, denoised, gt, si_psnr_input, si_psnr_pred)

        print(f"{source_file.name}: SI-PSNR {si_psnr_input:.2f} -> {si_psnr_pred:.2f}, "
              f"MicroSSIM {ssim_input:.4f} -> {ssim_pred:.4f}  (input -> prediction)")

label = "Mean over QC images" if len(scores["si_psnr_pred"]) > 1 else "QC image"
print(f"\n{'':32}{'input':>10}{'prediction':>13}{'gain':>10}")
print(f"{label + ', SI-PSNR:':32}"
      f"{np.mean(scores['si_psnr_input']):>10.2f}{np.mean(scores['si_psnr_pred']):>13.2f}"
      f"{np.mean(scores['si_psnr_pred']) - np.mean(scores['si_psnr_input']):>+10.2f}")
print(f"{label + ', MicroSSIM:':32}"
      f"{np.mean(scores['ssim_input']):>10.4f}{np.mean(scores['ssim_pred']):>13.4f}"
      f"{np.mean(scores['ssim_pred']) - np.mean(scores['ssim_input']):>+10.4f}")
print(f"\nMetrics and error maps saved to: {qc_dir}")

# Display the first QC image: full field of view on top, a zoomed-in crop underneath.
noisy, denoised, gt, si_psnr_input, si_psnr_pred = first_image
height, width = gt.shape[-2:]
half = min(int(qc_crop_size), height, width) // 2
rows = slice(height // 2 - half, height // 2 + half)
cols = slice(width // 2 - half, width // 2 + half)

panels = [
    (noisy, f"Noisy input\nSI-PSNR: {si_psnr_input:.2f}"),
    (denoised, f"Prediction\nSI-PSNR: {si_psnr_pred:.2f}"),
    (gt, "Ground truth"),
]
fig, ax = plt.subplots(2, 3, figsize=(15, 10))
for column, (image, title) in enumerate(panels):
    ax[0, column].imshow(image, cmap="gray")
    ax[0, column].set_title(title)
    ax[1, column].imshow(image[rows, cols], cmap="gray")
for axis in ax.ravel():
    axis.axis("off")
plt.show()


# **6. Using the trained model**
---

## **6.1. Generate predictions from an unseen dataset**

In [ ]:
#@markdown ###Path to the data to denoise and the folder where results are saved:
Data_folder = "example_data_n2n/prediction" #@param {type:"string"}
Result_folder = "example_data_n2n/results" #@param {type:"string"}

data_path = require_path(Data_folder, "Data_folder")
result_dir = require_path(Result_folder, "Result_folder", must_exist=False)
result_dir.mkdir(parents=True, exist_ok=True)

predictions, sources = careamist.predict(
    pred_data=data_path,
    tile_size=(prediction_tile_size, prediction_tile_size),
)

for pred, source in zip(predictions, sources):
    out_name = Path(source).stem + "_denoised.tif"
    out_path = result_dir / out_name
    tifffile.imwrite(out_path, np.asarray(pred).squeeze().astype(np.float32))
    print(f"Saved: {out_path}")


## **6.2. Assess the predicted output**

In [ ]:
#@markdown ##Display an input image next to its denoised prediction
idx = 0 #@param {type:"number"}
idx = int(idx)

if not predictions:
    raise RuntimeError("No predictions are available. Run section 6.1 first.")
if idx < 0 or idx >= len(predictions):
    raise IndexError(f"idx must be between 0 and {len(predictions) - 1}.")

# `sources` comes from CAREamics itself, so the input always matches the prediction.
input_img = tifffile.imread(sources[idx])
pred_img = np.asarray(predictions[idx]).squeeze()

fig, ax = plt.subplots(1, 2, figsize=(12, 6))
ax[0].imshow(input_img, cmap="gray")
ax[0].set_title("Input (noisy)")
ax[1].imshow(pred_img, cmap="gray")
ax[1].set_title("Prediction (denoised)")
for axis in ax:
    axis.axis("off")
fig.suptitle(Path(sources[idx]).name)
plt.show()


## **6.3. Download your predictions**
---

<font size = 4>**Store your data** and ALL its results elsewhere by downloading them from your Google Drive, and then clean up the original folder tree (dataset, results, trained model) if you plan to train or use another network. Please note that the notebook will otherwise **OVERWRITE** all files which have the same name.

# **7. Version log**
---
<font size = 4>**v1.0 (CAREamics)**:
*   First release of the CAREamics-powered Noise2Noise 2D notebook.
*   Built on CAREamics 0.3.2 (`create_n2n_config` + `CAREamist`), replacing the TensorFlow/Keras training stack.
*   Section 5.2 reports scale-invariant PSNR and MicroSSIM for the prediction and for the noisy input against the ground truth, saves MicroSSIM maps, and writes `QC_metrics_<model_name>.csv`.

# **Thank you for using Noise2Noise 2D!**